In [6]:
import os
import sys
from pathlib import Path

# Pin the repo so imports never hit another clone (e.g. PyCharmProjects). Options:
# - Set EXPLICIT_REPO_ROOT to this path, or "" to ignore.
# - Or set env MINI_HEDGE_ROOT before launching Jupyter.
# - If both unset, cwd must be repo root or scripts/ (see _resolve_root).
EXPLICIT_REPO_ROOT = "/Users/irenem/Desktop/Project/Mini_Hedge"
EXPLICIT_REPO_ROOT = (EXPLICIT_REPO_ROOT or "").strip()


def _resolve_root() -> Path:
    if EXPLICIT_REPO_ROOT:
        return Path(EXPLICIT_REPO_ROOT).resolve()
    env = (os.environ.get("MINI_HEDGE_ROOT") or "").strip()
    if env:
        return Path(env).resolve()
    _cwd = Path.cwd().resolve()
    if (_cwd / "mini_hedge").is_dir():
        return _cwd
    if (_cwd.parent / "mini_hedge").is_dir():
        return _cwd.parent
    raise RuntimeError(
        "Cannot find mini_hedge: set EXPLICIT_REPO_ROOT in this cell, or MINI_HEDGE_ROOT, "
        "or start Jupyter with cwd = this repo (folder that contains mini_hedge/)."
    )


_root = _resolve_root()
sys.path.insert(0, str(_root))

from mini_hedge import fetchers, storage, config
from mini_hedge.cli import cmd_snapshot

print(f"mini_hedge loaded from: {_root}")
print(f"FETCH_FULL_HISTORY = {config.FETCH_FULL_HISTORY}")
print(f"DEFAULT_DISPLAY_MONTHS = {config.DEFAULT_DISPLAY_MONTHS}")
print(f"DEFAULT_FETCH_MONTHS = {config.DEFAULT_FETCH_MONTHS} (window mode only)")
print(f"FRED key: {'set' if config.FRED_API_KEY else 'MISSING'}")
print(f"BLS key:  {'set' if config.BLS_API_KEY else 'MISSING'}")
print(f"DB path:  {config.DB_PATH}")

mini_hedge loaded from: /Users/irenem/Desktop/Project/Mini_Hedge
FETCH_FULL_HISTORY = True
DEFAULT_DISPLAY_MONTHS = 12
DEFAULT_FETCH_MONTHS = 120 (window mode only)
FRED key: set
BLS key:  set
DB path:  /Users/irenem/Desktop/Project/Mini_Hedge/data/econ.db


#### FRED — Fetch as DataFrame

In [7]:
# Full history when FETCH_FULL_HISTORY (default); else pass fetch_months=… for a window.
df_core_cpi = fetchers.fetch_fred("CPILFESL")
print(f"{len(df_core_cpi)} observations (full_history={config.FETCH_FULL_HISTORY}), dtypes:")
print(df_core_cpi.dtypes)
print()
df_core_cpi.head(6)

830 observations (full_history=True), dtypes:
date         datetime64[us]
value               float64
series_id               str
dtype: object



,date,value,series_id
0,2026-03-01,334.165,CPILFESL
1,2026-02-01,333.512,CPILFESL
2,2026-01-01,332.793,CPILFESL
3,2025-12-01,331.814,CPILFESL
4,2025-11-01,331.043,CPILFESL
5,2025-09-01,330.418,CPILFESL


#### BLS — Fetch as DataFrame

In [8]:
df_cpi = fetchers.fetch_bls("CUUR0000SA0")
print(f"{len(df_cpi)} observations (full_history={config.FETCH_FULL_HISTORY})")
print()
df_cpi.head(6)

1358 observations (full_history=True)



,date,value,footnotes,series_id
0,2026-03-01,330.213,,CUUR0000SA0
1,2026-02-01,326.785,,CUUR0000SA0
2,2026-01-01,325.252,,CUUR0000SA0
3,2025-12-01,324.054,,CUUR0000SA0
4,2025-11-01,324.122,,CUUR0000SA0
5,2025-09-01,324.800,,CUUR0000SA0


#### Query from SQLite (no API call needed)

In [9]:
# Query previously stored data from the local database — no API call
stored = storage.query_series("CUUR0000SA0", months=6)
print(f"{len(stored)} rows from SQLite archive:")
stored

6 rows from SQLite archive:


,series_id,date,value,source,footnotes,fetched_at
0,CUUR0000SA0,2026-03-01,330.213,bls,,2026-04-25T22:44:22.530898
1,CUUR0000SA0,2026-02-01,326.785,bls,,2026-04-25T22:44:22.530898
2,CUUR0000SA0,2026-01-01,325.252,bls,,2026-04-25T22:44:22.530898
3,CUUR0000SA0,2025-12-01,324.054,bls,,2026-04-25T22:44:22.530898
4,CUUR0000SA0,2025-11-01,324.122,bls,,2026-04-25T22:44:22.530898
5,CUUR0000SA0,2025-09-01,324.800,bls,,2026-04-25T22:44:22.530898


#### Snapshot — All Key Indicators

In [10]:
cmd_snapshot()


  US Economic Snapshot  —  2026-04-25 18:44
  9 rows  |  macro: SERIES_MAP + vol: VIX/MOVE

  Indicator                    Source  As of              Value   MoM/1D Δ%   YoY/252D Δ%   Z-Score
  ---------------------------  ------  ----------    ----------  ----------  ------------   -------
  CPI — All Urban Consumers    BLS     2026-03-01       330.213       1.049         3.488      1.54
  Core CPI — Ex Food & Energy  FRED    2026-03-01       334.165       0.196         2.673      1.47
  Federal Funds Rate           FRED    2026-03-01         3.640       0.000       -15.935      0.12
  10Y Treasury Yield           FRED    2026-04-14         4.260      -0.930        -1.843      0.70
  Yield Curve  (10Y − 2Y)      FRED    2026-04-14         0.500      -3.846        16.279      0.56
  Unemployment Rate            FRED    2026-03-01         4.300      -2.273         2.381      0.28
  Fed Funds Target — Upper     FRED    2026-04-17         3.750       0.000       -16.667     -1.77
  VIX (